# 用 Google Colab 訓練 YOLO 物件偵測模型

使用 [Ultralytics YOLO11](https://docs.ultralytics.com/) 在 **African Wildlife 資料集**上訓練物件偵測模型。
資料集有 4 個類別：水牛 (buffalo)、大象 (elephant)、犀牛 (rhino)、斑馬 (zebra)，約 1500 張圖片，第一次執行時會自動下載。

**使用方式：**
1. 上方選單「執行階段」→「變更執行階段類型」→ 選 **T4 GPU**
2. 「執行階段」→「全部執行」
3. 訓練約 10～20 分鐘，跑完最後一格會把模型下載回電腦

## 1. 確認 GPU

In [ ]:
!nvidia-smi

## 2. 安裝 Ultralytics

In [ ]:
!pip install -q ultralytics

import ultralytics
ultralytics.checks()

## 3. 開始訓練

- `yolo11n.pt`：最小的 YOLO11 預訓練模型，訓練快
- `african-wildlife.yaml`：資料集設定，會自動下載資料
- `epochs=30`：訓練 30 輪，想要更準可以調高

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")  # 載入預訓練模型

results = model.train(
    data="african-wildlife.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    project="runs",
    name="wildlife",
    exist_ok=True,
)

## 4. 訓練結果（Loss 曲線、mAP）

In [ ]:
from IPython.display import Image, display

save_dir = model.trainer.save_dir  # 訓練結果存放的資料夾
print("結果資料夾：", save_dir)
display(Image(filename=f"{save_dir}/results.png", width=900))

### 混淆矩陣

In [ ]:
display(Image(filename=f"{save_dir}/confusion_matrix_normalized.png", width=600))

## 5. 用驗證集評估模型

In [ ]:
best = YOLO(f"{save_dir}/weights/best.pt")  # 載入訓練好的最佳模型
metrics = best.val(data="african-wildlife.yaml")

print(f"mAP50    : {metrics.box.map50:.3f}")
print(f"mAP50-95 : {metrics.box.map:.3f}")

## 6. 拿測試圖片做預測

In [ ]:
import glob, random
from PIL import Image as PILImage

test_images = sorted(glob.glob("/content/datasets/african-wildlife/images/test/*.jpg"))
random.seed(0)
samples = random.sample(test_images, 6)  # 隨機挑 6 張測試圖片

preds = best.predict(samples)

for p in preds:
    img = PILImage.fromarray(p.plot()[..., ::-1])  # 畫上偵測框
    img.thumbnail((500, 500))
    display(img)

## 7. 下載訓練好的模型

In [ ]:
import shutil
from google.colab import files

shutil.copy(f"{save_dir}/weights/best.pt", "/content/best.pt")
shutil.make_archive("/content/wildlife_results", "zip", save_dir)

files.download("/content/best.pt")
files.download("/content/wildlife_results.zip")